# Adaptive LayerNorm Zero (adaLN-Zero)

**难度：** Medium

**函数签名：** `adaln_zero(x, cond, W_ada, b_ada) -> Tensor`

**参数：**
- `x` — 输入 token (B, N, D)
- `cond` — 条件嵌入 (B, D)
- `W_ada` — 线性权重 (D, 6*D)
- `b_ada` — 偏置 (6*D,)

**返回：** 调制后的张量 (B, N, D)

**核心思想：** DiT 用条件嵌入回归 LayerNorm 的缩放/偏移/门控参数

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def adaln_zero(x, cond, W_ada, b_ada):
    B, N, D = x.shape

    # ---- Step 1: 从条件嵌入回归调制参数 ----
    # cond @ W_ada: (B, D) @ (D, 6*D) → (B, 6*D)
    # + b_ada: 广播加偏置
    # 6*D 维度包含 6 个参数组: [γ1, β1, α1, γ2, β2, α2]
    # γ: 缩放（scale）  β: 偏移（shift）  α: 门控（gate）
    # 我们只用第一组 (γ1, β1, α1) 来调制
    params = cond @ W_ada + b_ada

    # ---- Step 2: 切分为 6 个参数 ----
    # chunk(6, dim=-1) 将最后一维均匀切成 6 份
    # 每份大小 D
    chunks = params.chunk(6, dim=-1)
    gamma1, beta1, alpha1 = chunks[0], chunks[1], chunks[2]
    # gamma1, beta1, alpha1 各自形状: (B, D)

    # ---- Step 3: 手动 LayerNorm（无可学习仿射参数）----
    # 标准 LN: y = (x - mean) / sqrt(var + eps) * gamma + beta
    # 这里的 LN 没有内置的 gamma/beta，因为我们要用条件参数替代
    # dim=-1: 对特征维做归一化
    # keepdim=True: 保持维度以便广播
    # unbiased=False: 用总体方差（除以 N 而非 N-1）
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / (var + 1e-6).sqrt()

    # ---- Step 4: 条件调制 ----
    # unsqueeze(1): (B, D) → (B, 1, D) 以便与 (B, N, D) 广播
    # out = α * (γ * x_norm + β)
    # α 是门控参数: 零初始化时 α=0，整个模块输出为 0（恒等映射）
    # 这是 adaLN-Zero 的关键设计: 初始时不影响网络输出
    out = alpha1.unsqueeze(1) * (gamma1.unsqueeze(1) * x_norm + beta1.unsqueeze(1))
    return out

In [ ]:
torch.manual_seed(0)

B, N, D = 4, 16, 32
C = 16  # conditioning dim

x    = torch.randn(B, N, D)
cond = torch.randn(B, C)

# Zero W_ada => all params = b_ada; if b_ada is also zero => alpha1=0 => output is zero
W_ada = torch.zeros(C, 6 * D)
b_ada = torch.zeros(6 * D)
out_zero = adaln_zero(x, cond, W_ada, b_ada)
print(f"Zero W_ada, zero b_ada => max abs output: {out_zero.abs().max().item():.6f}  (expected 0.0)")

# Non-zero W_ada => output should be non-zero
W_ada_rand = torch.randn(C, 6 * D) * 0.1
b_ada_rand = torch.randn(6 * D) * 0.1
out_rand = adaln_zero(x, cond, W_ada_rand, b_ada_rand)
print(f"Random W_ada          => output shape: {out_rand.shape}, mean abs: {out_rand.abs().mean().item():.4f}")

In [ ]:
from torch_judge import check
check("adaln_zero")

## 📝 核心思路总结

1. **adaLN-Zero**：条件嵌入回归 LayerNorm 的 γ/β/α 参数
2. **零初始化门控**：α 初始为 0，模块初始输出为 0（恒等映射）
3. **手动 LayerNorm**：减均值除标准差，无可学习参数
4. **chunk 切分**：`(B, 6D)` 切成 6 个 `(B, D)` 参数组